Samuel Scott - Module 2 Homework - Takyo Software Cataloger

In [29]:
#!pip install mlxtend
#!pip install xgboost
#!pip install scikit-plot


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
#Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as sm
from mlxtend.feature_selection import ExhaustiveFeatureSelector, SequentialFeatureSelector
from sklearn.linear_model import BayesianRidge, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import scipy
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, roc_curve, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
import warnings
warnings.filterwarnings("ignore")
import mlba

## Data Sanity Check

In [3]:
tayko_data = pd.read_csv('C:/Users/samsc/Desktop/ADS-505/Tayko.csv')
tayko_data.head()

,sequence_number,US,source_a,source_c,source_b,source_d,source_e,source_m,source_o,source_h,...,source_x,source_w,Freq,last_update_days_ago,1st_update_days_ago,Web order,Gender=male,Address_is_res,Purchase,Spending
0,1,1,0,0,1,0,0,0,0,0,...,0,0,2,3662,3662,1,0,1,1,128
1,2,1,0,0,0,0,1,0,0,0,...,0,0,0,2900,2900,1,1,0,0,0
2,3,1,0,0,0,0,0,0,0,0,...,0,0,2,3883,3914,0,0,0,1,127
3,4,1,0,1,0,0,0,0,0,0,...,0,0,1,829,829,0,1,0,0,0
4,5,1,0,1,0,0,0,0,0,0,...,0,0,1,869,869,0,0,0,0,0


In [4]:
tayko_data.shape

(2000, 25)

In [5]:
#No missing data
tayko_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   sequence_number       2000 non-null   int64
 1   US                    2000 non-null   int64
 2   source_a              2000 non-null   int64
 3   source_c              2000 non-null   int64
 4   source_b              2000 non-null   int64
 5   source_d              2000 non-null   int64
 6   source_e              2000 non-null   int64
 7   source_m              2000 non-null   int64
 8   source_o              2000 non-null   int64
 9   source_h              2000 non-null   int64
 10  source_r              2000 non-null   int64
 11  source_s              2000 non-null   int64
 12  source_t              2000 non-null   int64
 13  source_u              2000 non-null   int64
 14  source_p              2000 non-null   int64
 15  source_x              2000 non-null   int64
 16  source

## The Company

Tayko is a software catalog firm that sells games and educational software. It started as a software
manufacturer and later expanded its offerings by adding third-party titles. Tayko has recently assembled a
revised collection of items in a new catalog, which it is preparing to roll out in a large mailing campaign.
Tayko's customer list is a key strategic asset. To expand its customer base, Tayko has joined a
consortium of catalog firms that specialize in computer and software products. Consortium members pool
their customer lists and can withdraw an equivalent number of names each quarter for their own mailings.
Members are allowed to apply predictive modeling to the pooled records so they can select names more
effectively, mailing to those most likely to respond rather than mailing to the list at random

## The Mailing Experiment

Tayko supplied its 200,000-name customer list to the consortium pool, which now totals over 5,000,000
names. This entitles Tayko to draw 200,000 names for an upcoming mailing. To improve the odds of
selecting high-performing prospects, Tayko first ran a test by drawing 20,000 names from the pool and
mailing the new catalog to that test sample.
The test mailing produced 1,065 purchasers, a response rate of 5.3%. To improve the signal available to
the modeling techniques, the working dataset was constructed as a stratified sample with equal numbers
of purchasers and non-purchasers (1,000 each), producing an apparent response rate of 50%. After
modeling, the predicted probabilities must be adjusted back to the true population rate by multiplying each
case's probability of purchase by 0.053 / 0.5 = 0.107.

### Question 1 - Baseline Profit Estimate
Each catalog costs approximately $2 to mail (including printing, postage, and mailing costs). Estimate the
gross profit that Tayko could expect from the remaining 180,000 names if it selects them randomly from
the pool. This becomes your business baseline. Every model later in the assignment must be compared
against it

In [6]:
#Baseline model
num_of_names = 180000
cost = num_of_names * 2 # $360,000 cost
response_rate = 0.053 #5.3% in decimal form
avg_spender = tayko_data['Spending'].mean().round(2) #average customer spends $102.62
earnings = (num_of_names * response_rate) * avg_spender #the earnings are the customers guaranteed to respond times the amount the avg_spender spends
gross_profit = earnings - cost 
gross_profit

np.float64(618994.8)

The baseline model projects the gross profit to be $618,994.8

### Question 2 - Classification Model: Purchaser vs. Non-Purchaser
Develop a model for classifying a customer as a purchaser or non-purchaser.

#### 2.1 Partition the Data
Partition the data randomly into a training set (800 records), validation set (700 records), and test set (500
records). Use a fixed random seed for reproducibility.

In [7]:
#Partition training vs test
outcome = 'Purchase'
predictors = tayko_data.drop(columns=['sequence_number', 'Spending', 'Purchase'])
predictors = predictors.columns

X = tayko_data[predictors]
y = tayko_data[outcome]

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=1)

print(train_X.shape)
print(test_X.shape)
print(train_y.shape)
print(test_y.shape)

(1500, 22)
(500, 22)
(1500,)
(500,)


In [8]:
#Partition training vs holdout
train_X, holdout_X, train_y, holdout_y = train_test_split(train_X, train_y, test_size=700/1500, random_state=1)
print(train_X.shape)
print(holdout_X.shape)
print(train_y.shape)
print(holdout_y.shape)
print(test_X.shape)
print(test_y.shape)

(800, 22)
(700, 22)
(800,)
(700,)
(500, 22)
(500,)


#### 2.2 Logistic Regression with L2 Penalty
Using only the **training set**, run a cross-validated logistic regression with an L2 penalty using
LogisticRegressionCV with parameters solver='lbfgs', cv=5, and max_iter=500. Use this
model to classify the data into purchasers and non-purchasers. Logistic regression is chosen here
because it yields an estimated probability of purchase, which is required later when you compute
expected spending.

In [9]:
log_reg = LogisticRegressionCV(cv=5, max_iter=500, solver='lbfgs', l1_ratios=None, scoring=None, use_legacy_attributes=True, random_state=1)
log_reg.fit(train_X, train_y)

,"l1_ratios l1_ratios: array-like of shape (n_l1_ratios), default=NoneFloats between 0 and 1 passed as Elastic-Net mixing parameter (scaling betweenL1 and L2 penalties). For `l1_ratio = 0` the penalty is an L2 penalty. For`l1_ratio = 1` it is an L1 penalty. For `0 < l1_ratio < 1`, the penalty is acombination of L1 and L2.All the values of the given array-like are tested by cross-validation and theone giving the best prediction score is used... warning:: Certain values of `l1_ratios`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `l1_ratios=None` is deprecated in 1.8 and will raise an error in version 1.10. Default value will change from `None` to `(0.0,)` in version 1.10.",None
,"cv cv: int or cross-validation generator, default=NoneThe default cross-validation generator used is Stratified K-Folds.If an integer is provided, it specifies the number of folds, `n_folds`, used.See the module :mod:`sklearn.model_selection` module for thelist of possible cross-validation objects... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str or callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: :ref:`accuracy <accuracy_score>` is used... versionchanged:: 1.11 The default will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11.",None
,"max_iter max_iter: int, default=100Maximum number of iterations of the optimization algorithm.",500
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.Note that this only applies to the solver and not the cross-validationgenerator. See :term:`Glossary <random_state>` for details.",1
,"use_legacy_attributes use_legacy_attributes: bool, default=TrueIf True, use legacy values for attributes:- `C_` is an ndarray of shape (n_classes,) with the same value repeated- `l1_ratio_` is an ndarray of shape (n_classes,) with the same value repeated- `coefs_paths_` is a dict with class labels as keys and ndarrays as values- `scores_` is a dict with class labels as keys and ndarrays as values- `n_iter_` is an ndarray of shape (1, n_folds, n_cs) or similarIf False, use new values for attributes:- `C_` is a float- `l1_ratio_` is a float- `coefs_paths_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs, n_classes, n_features) For binary problems (n_classes=2), the 2nd last dimension is 1.- `scores_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs)- `n_iter_` is an ndarray of shape (n_folds, n_l1_ratios, n_cs).. versionchanged:: 1.10 The default will change from True to False in version 1.10... deprecated:: 1.10 `use_legacy_attributes` will be deprecated in version 1.10 and be removed in 1.12.",True
,"Cs Cs: int or list of floats, default=10Each of the values in Cs describes the inverse of regularizationstrength. If Cs is as an int, then a grid of Cs values are chosenin a logarithmic scale between 1e-4 and 1e4.Like in support vector machines, smaller values specify strongerregularization.",10
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer dual=False whenn_samples > n_features.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet'}, default='l2'Specify the norm of the penalty:- `'l2'`: add an L2 penalty term (used by default);- `'l1'`: add an L1 penalty term;- `'elasticnet'

In [10]:
#Making sure 
print(log_reg.classes_)
print(log_reg.coef_) #a coefficient for each predictor
print(log_reg.intercept_)
print(log_reg.scores_) #CV scores for each C score which defaulted to 10

[0 1]
[[ 3.39393940e-01  1.53140552e+00 -7.28883745e-01 -1.57332223e-01
   8.51739010e-01  4.68651202e-01  7.19150274e-01  3.46851851e-01
  -4.24837736e+00  3.29037889e-01 -1.40301560e-01  1.16224076e+00
   1.69809848e+00  1.46070810e+00  6.95513948e-01  1.14354160e+00
   2.27091595e+00  2.92190765e-04 -3.84897487e-04  7.57165651e-01
  -3.10510273e-01 -6.99585135e-01]]
[-3.34006075]
{np.int64(1): array([[0.6375 , 0.65   , 0.75625, 0.80625, 0.79375, 0.79375, 0.8    ,
        0.79375, 0.79375, 0.79375],
       [0.64375, 0.65   , 0.7125 , 0.74375, 0.775  , 0.775  , 0.78125,
        0.78125, 0.78125, 0.78125],
       [0.6375 , 0.65   , 0.7375 , 0.8125 , 0.85   , 0.83125, 0.825  ,
        0.83125, 0.83125, 0.8375 ],
       [0.6375 , 0.65   , 0.7125 , 0.76875, 0.8125 , 0.8375 , 0.83125,
        0.81875, 0.825  , 0.825  ],
       [0.625  , 0.65   , 0.725  , 0.7875 , 0.8375 , 0.85   , 0.85   ,
        0.8375 , 0.84375, 0.8375 ]])}


In [11]:
#predict_proba() is used to predict probabilities while predict() gives the appropriate class label. In this case, 
#we want to use predict_proba()
log_reg_probs = log_reg.predict_proba(test_X)[:, 1] # what is the probability of the customer purchasing

In [12]:
#never mind we need the predict() version as well due to question 3?
log_reg_class = log_reg.predict(test_X) #the cutoff is default 0.5 because this is binary classification

### Question 3 - Prediction Model: Spending Among Purchasers
Develop a model for predicting spending among the purchasers.
#### 3.1 Filter to Purchasers
Create subsets of the training and validation sets containing only purchasers by filtering for Purchase = 1. The spending model is trained on actual buyers only.

In [13]:
#have all the purchasers filtered on each partition
train_purchase_mask = train_X[train_y == 1] #out of the training variables which ones are purchasers and so on for the other 2 sets
holdout_purchase_mask = holdout_X[holdout_y == 1]
test_purchase_mask = test_X[test_y == 1]

In [14]:
spending = tayko_data['Spending'] #just get the spending column
train_purchase_spending = spending.loc[train_X.index][train_y == 1] #get only the indexes values in the training data where purchased customers are selected
holdout_purchase_spending = spending.loc[holdout_X.index][holdout_y == 1]
test_purchase_spending = spending.loc[test_X.index][test_y == 1]

#### 3.2 Build and Compare Two Spending Models
Develop two models for predicting spending using the filtered datasets, then choose between them.
1. Multiple linear regression with stepwise variable selection
2. Regression tree
3. Choose one model on the basis of its performance on the validation data. Explain your
reasoning for selecting it.

In [15]:
#Stepwise regression model
model1 = LinearRegression()
sfs_stepwise = SequentialFeatureSelector(model1,
                                         forward=True, floating=True,
                                         cv=5, scoring='neg_root_mean_squared_error',
                                         n_jobs=-1)
sfs_stepwise.fit(train_purchase_mask, train_purchase_spending)

,estimator,LinearRegression()
,k_features,"(1, ...)"
,floating,True
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,forward,True
,verbose,0
,cv,5
,pre_dispatch,'2*n_jobs'
,clone_estimator,True
,fixed_features,None


In [16]:
#We used the stepwise selection algorithm to find best subset 
best_subset = sfs_stepwise.subsets_[1] 
for v in sfs_stepwise.subsets_.values():
    if v['avg_score'] > best_subset['avg_score']:
        best_subset = v

print(f"Best accuracy score: {- best_subset['avg_score']:.2f}")
print(f"Best subset (indices): {best_subset['feature_idx']}")
print(f"Best subset (feat_name): {best_subset['feature_names']}")

Best accuracy score: 159.80
Best subset (indices): (16,)
Best subset (feat_name): ('Freq',)


In [17]:
selected_features = list(best_subset['feature_names'])
model1.fit(train_purchase_mask[selected_features],train_purchase_spending)
model1_pred = model1.predict(test_purchase_mask[selected_features])

In [18]:
#Regression tree
param_dist = {
     "max_depth": list(range(1,25)),
     "min_impurity_decrease": scipy.stats.expon(scale=1),
     "min_samples_split": list(range(2,50)),
 }

model2 = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=1), param_dist, cv=5, n_iter=200, n_jobs=-1, random_state=1)

In [19]:
model2.fit(
train_purchase_mask,train_purchase_spending)
model2_pred = model2.predict(test_purchase_mask)

In [20]:
#I had AI help me create this adjusted R^2 function
# r2: standard R2 score
# n: number of observations (rows)
# p: number of independent variables (features)
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * ((n - 1) / (n - p - 1))

In [21]:
stepwise_results = pd.DataFrame(
        {'Stepwise_Model_RMSE': [root_mean_squared_error(test_purchase_spending, model1_pred)],
         'Stepwise_Model_MAE': [mean_absolute_error(test_purchase_spending, model1_pred)],
         'Stepwise_Model_AdjR^2': [adjusted_r2(r2_score(test_purchase_spending, model1_pred),len(test_purchase_spending),test_purchase_mask.shape[1])],
        })
print(stepwise_results)
regression_tree_results = pd.DataFrame({
         'Regression_Tree_RMSE': [root_mean_squared_error(test_purchase_spending, model2_pred)],
         'Regression_Tree_MAE': [mean_absolute_error(test_purchase_spending, model2_pred)],
         'Regression_Tree_AdjR^2': [adjusted_r2(r2_score(test_purchase_spending, model2_pred),len(test_purchase_spending),test_purchase_mask.shape[1])]
        })
print(regression_tree_results)

   Stepwise_Model_RMSE  Stepwise_Model_MAE  Stepwise_Model_AdjR^2
0            200.68226           120.96295               0.286894
   Regression_Tree_RMSE  Regression_Tree_MAE  Regression_Tree_AdjR^2
0            205.710312           116.692746                0.250713


The model of choice I decided to go with is the Stepwise model because it has 2 out of the 3 metrics that perform better than the Regression Tree. Both RMSE and R^2 were better for the stepwise model which to me is indicative of a better model choice because it will more accurately predict how much customers will spend when they make purchase(s). Having a lower RMSE means that predictions made on the data are closer to the actual result. Having a higher R^2 score means that proportion of the predictors variability is explained better for the model. 

### Question 4 - Score Analysis and Profit Estimate
Return to the original test data partition, which contains both purchasers and non-purchasers. Create a
new data frame called Score_Analysis containing the test data portion of the dataset.

In [22]:
Score_Analysis = pd.concat([test_X, test_y], axis=1)
Score_Analysis.head()

,US,source_a,source_c,source_b,source_d,source_e,source_m,source_o,source_h,source_r,...,source_p,source_x,source_w,Freq,last_update_days_ago,1st_update_days_ago,Web order,Gender=male,Address_is_res,Purchase
674,1,0,0,0,0,0,0,0,1,0,...,0,0,0,2,1346,1386,0,1,1,0
1699,1,0,0,0,0,0,0,0,0,0,...,0,0,1,2,434,463,1,0,0,1
1282,1,1,0,0,0,0,0,0,0,0,...,0,0,0,1,3717,3717,0,1,0,0
1315,1,1,0,0,0,0,0,0,0,0,...,0,0,0,9,1128,3376,1,1,0,1
1210,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,2765,2765,1,0,0,0


#### 4.1 Add Predicted Purchase Probability
Add a column with the predicted scores (probability of purchase) from the logistic regression model.

In [23]:
Score_Analysis['Prob_of_Purch'] = log_reg_probs 

#### 4.2 Add Predicted Spending
Add another column with the predicted spending amount from the spending model you selected in
Question 3.2

In [24]:
Score_Analysis['Predicted_Spending'] = 0.00 #impute purchasers = 0 with 0 spending

Score_Analysis.loc[test_purchase_mask.index, 'Predicted_Spending'] = model1_pred.round(2)

Score_Analysis

,US,source_a,source_c,source_b,source_d,source_e,source_m,source_o,source_h,source_r,...,source_w,Freq,last_update_days_ago,1st_update_days_ago,Web order,Gender=male,Address_is_res,Purchase,Prob_of_Purch,Predicted_Spending
674,1,0,0,0,0,0,0,0,1,0,...,0,2,1346,1386,0,1,1,0,0.020684,0.00
1699,1,0,0,0,0,0,0,0,0,0,...,1,2,434,463,1,0,0,1,0.967406,201.53
1282,1,1,0,0,0,0,0,0,0,0,...,0,1,3717,3717,0,1,0,0,0.536573,0.00
1315,1,1,0,0,0,0,0,0,0,0,...,0,9,1128,3376,1,1,0,1,1.000000,877.12
1210,0,0,0,0,0,1,0,0,0,0,...,0,0,2765,2765,1,0,0,0,0.085444,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,1,0,0,0,0,0,0,0,0,0,...,0,1,1184,3376,0,1,0,1,0.214467,105.01
1450,1,1,0,0,0,0,0,0,0,0,...,0,2,1135,2121,1,1,0,1,0.954108,201.53
1919,0,0,0,0,0,1,0,0,0,0,...,0,3,2296,2564,1,0,0,1,0.987660,298.04
255,1,0,0,0,0,0,0,0,1,0,...,0,2,1415,1436,0,1,1,0,0.020703,0.00


#### 4.3 Compute Expected Spending
Add a column for expected spending, calculated as the adjusted probability of purchase multiplied by
predicted spending. Remember to apply the 0.107 adjustment factor (0.053 / 0.5) to convert the apparent
probability from the stratified sample back to the true population rate.

In [25]:
Score_Analysis['Expected_Spending'] = Score_Analysis['Prob_of_Purch'] * 0.107 * Score_Analysis['Predicted_Spending']
Score_Analysis

,US,source_a,source_c,source_b,source_d,source_e,source_m,source_o,source_h,source_r,...,Freq,last_update_days_ago,1st_update_days_ago,Web order,Gender=male,Address_is_res,Purchase,Prob_of_Purch,Predicted_Spending,Expected_Spending
674,1,0,0,0,0,0,0,0,1,0,...,2,1346,1386,0,1,1,0,0.020684,0.00,0.000000
1699,1,0,0,0,0,0,0,0,0,0,...,2,434,463,1,0,0,1,0.967406,201.53,20.860853
1282,1,1,0,0,0,0,0,0,0,0,...,1,3717,3717,0,1,0,0,0.536573,0.00,0.000000
1315,1,1,0,0,0,0,0,0,0,0,...,9,1128,3376,1,1,0,1,1.000000,877.12,93.851839
1210,0,0,0,0,0,1,0,0,0,0,...,0,2765,2765,1,0,0,0,0.085444,0.00,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,1,0,0,0,0,0,0,0,0,0,...,1,1184,3376,0,1,0,1,0.214467,105.01,2.409765
1450,1,1,0,0,0,0,0,0,0,0,...,2,1135,2121,1,1,0,1,0.954108,201.53,20.574117
1919,0,0,0,0,0,1,0,0,0,0,...,3,2296,2564,1,0,0,1,0.987660,298.04,31.496756
255,1,0,0,0,0,0,0,0,1,0,...,2,1415,1436,0,1,1,0,0.020703,0.00,0.000000


#### 4.4 Cumulative Gains Chart
Plot the cumulative gains chart of expected spending: cumulative expected spending on the y-axis,
number of records targeted (sorted by expected spending, descending) on the x-axis. Include a reference
line for random selection.

In [40]:
sorted_a = Score_Analysis.sort_values(by='Expected_Spending',ascending=False)
mlba.gainsChart(Score_Analysis, ranking='Expected_Spending', actual=test_purchase_mask)

ValueError: Boolean array expected for the condition, not object